# Fig 3b - treatment convergence (ORA)

Over-representation analysis (ORA) of the KO-vs-control hits per lineage - the same
direction-agnostic enrichment used in the Fig 1 main panels (the treatment GSEA, which
ranks by unsigned -log10 chi-sq p and does NOT use delta-psi, is the Fig 3 full-enrichment supplement).
Curated convergent themes; dot size = overlapping gene count, colour = -log10(padj). The
SFARI-not-enriched result and the RBP term (ORA-nonsignificant) are described in the text.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL = "fig3b_convergence_ora"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, 780  # 85 x 66 mm (half-grid)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)
KB = (
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/dirmult_treatment_results"
)

# (Description substring, theme, display label) - terms that are actually ORA-significant.
# The cross-lineage convergence is the translation machinery; Myc/NMD lean inhibitory.
CURATED = [
    ("cytoplasmic translation", "Translation", "cytoplasmic translation"),
    ("Cap-dependent Translation Initiation", "Translation", "translation initiation"),
    ("Eukaryotic Translation Elongation", "Translation", "translation elongation"),
    ("Ribosome", "Translation", "ribosome (KEGG)"),
    ("MYC_TARGETS_V1", "Growth", "Myc targets V1"),
    ("Nonsense-Mediated Decay (NMD)", "mRNA decay", "NMD (Reactome)"),
]

In [ ]:
o = pd.read_csv(f"{KB}/enrichment_discovery_top_ora.csv")
o = o[o["kind"] == "ORA"] if "kind" in o.columns else o


def overlap_count(gr):
    try:
        return int(str(gr).split("/")[0])
    except Exception:
        return np.nan


recs = []
for substr, theme, label in CURATED:
    for lin in ["excitatory", "inhibitory"]:
        sub = o[
            (o["lineage"] == lin)
            & (o["Description"].str.contains(substr, case=False, regex=False))
        ]
        if len(sub):
            r = sub.nsmallest(1, "p.adjust").iloc[0]
            recs.append(
                dict(
                    theme=theme,
                    label=label,
                    lineage=lin,
                    padj=float(r["p.adjust"]),
                    k=overlap_count(r.get("GeneRatio", np.nan)),
                )
            )
D = pd.DataFrame(recs)
print(D.to_string())

In [ ]:
terms, seen = [], set()
for _, theme, label in CURATED:
    if label not in seen:
        terms.append((theme, label))
        seen.add(label)
ypos, y, last = {}, 0.0, None
for theme, label in terms:
    if last is not None and theme != last:
        y += 0.7
    ypos[label] = y
    y += 1.0
    last = theme
ymax = y

xmap = {"excitatory": 0.0, "inhibitory": 1.0}
fig, ax = plt.subplots(figsize=FIGSIZE)
nlp = -np.log10(D["padj"].clip(lower=1e-300))
sizes = 6 + 26 * (nlp - nlp.min()).clip(lower=0) / (nlp.max() - nlp.min() + 1e-9)
sc = ax.scatter(
    [xmap[l] for l in D["lineage"]],
    [ypos[lb] for lb in D["label"]],
    s=sizes,
    c=nlp,
    cmap="OrRd",
    edgecolors="black",
    linewidths=0.3,
    zorder=3,
)

ax.set_yticks([])
for theme, lb in terms:
    ax.text(-0.16, ypos[lb], lb, fontsize=S.FONT_TICK, va="center", ha="right")
THEMES = ["Translation", "Growth", "mRNA decay"]
for theme in THEMES:
    ys = [ypos[lb] for th, lb in terms if th == theme]
    ax.text(
        -1.16,
        np.mean(ys),
        theme,
        fontsize=S.FONT_TICK,
        fontweight="bold",
        color="#777777",
        va="center",
        ha="center",
        rotation=90,
    )
ax.set_xticks([0, 1])
ax.set_xticklabels(["Excitatory", "Inhibitory"], fontsize=S.FONT_LABEL)
for t, lin in zip(ax.get_xticklabels(), ["excitatory", "inhibitory"]):
    t.set_color(S.paper_palette_lineage[lin])
    t.set_fontweight("bold")
ax.set_xlim(-1.30, 1.6)
ax.set_ylim(-0.7, ymax - 0.3)
ax.invert_yaxis()
ax.tick_params(length=0)
for sp in ["top", "right", "left", "bottom"]:
    ax.spines[sp].set_visible(False)

cb = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
cb.set_label(r"$-\log_{10}(p_{\mathrm{adj}})$", fontsize=S.FONT_TICK)
cb.ax.tick_params(labelsize=S.FONT_TICK - 1)
size_leg = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="0.6",
        markeredgecolor="black",
        markeredgewidth=0.3,
        markersize=np.sqrt(6 + 26 * f),
        label=lab,
    )
    for f, lab in [(0.0, "low"), (0.5, "  "), (1.0, "high")]
]
ax.legend(
    handles=size_leg,
    title=r"$-\log_{10}p_{\mathrm{adj}}$",
    fontsize=S.FONT_LEGEND - 0.5,
    title_fontsize=S.FONT_LEGEND,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.04),
    ncol=3,
    columnspacing=0.8,
    handletextpad=0.2,
)
fig.subplots_adjust(left=0.05, right=0.88, top=0.95, bottom=0.16)
S.save_and_show(fig, PANEL, figure="Fig3")

## Variants: quarter-height resize + convergence/divergence (treatment, chisq ORA)

**(1)** `fig3b_convergence_ora__half_quarter` - the same treatment-convergence dotplot at
half-width x quarter-height with tighter spacing and left database-theme strips. **(2)**
`fig3b_convergence_divergence` - convergence AND divergence between the excitatory and inhibitory
treatment responses (chisq ORA; treatment has no permutation null so no intersection here). Groups:
**Convergent** (significant in both: translational machinery, NMD, axon guidance), **Excitatory-
specific** (glutamatergic/NMDA-receptor signalling, RHO-GTPase/cytoskeletal, Sema3A axon repulsion),
**Inhibitory-specific** (Myc targets, growth-factor-receptor signalling). Open circles = n.s.
(padj>=0.05). Note the divergence is asymmetric: the excitatory lineage carries a much larger
lineage-specific synaptic program, while the inhibitory-specific signal is narrower (growth/Myc).


In [ ]:
# --- shared convergence/divergence dotplot (tight layout: close columns, small axis labels,
# left group strips sized to fit 2-line labels). rows: (group, display_label, padj_exc, padj_inh). ---
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

_EXC_X, _INH_X = 0.0, 0.6
_STRIP_X, _STRIP_W = -1.60, 0.17
_LABEL_X, _XLIM, _GAP = -0.10, (-1.82, 0.95), 0.7


def _convergence_dotplot(rows, gcol, save, figure, allow_ns=True, short_cbar=False):
    ypos, y, last, grp_rows = {}, 0.0, None, {}
    for grp, disp, pe, pi in rows:
        if last is not None and grp != last:
            y += _GAP
        ypos[disp] = y
        grp_rows.setdefault(grp, []).append(y)
        y += 1.0
        last = grp
    ymax = y
    allp = [p for _, _, pe, pi in rows for p in (pe, pi) if p == p and p < 0.05]
    nlp_all = -np.log10(np.array(allp))
    lo, hi = nlp_all.min(), nlp_all.max()
    fig, ax = plt.subplots(figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
    for grp, disp, pe, pi in rows:
        for x, p in [(_EXC_X, pe), (_INH_X, pi)]:
            yy = ypos[disp]
            if p == p and p < 0.05:
                nlp = -np.log10(p)
                s = 7 + 24 * ((nlp - lo) / (hi - lo + 1e-9))
                ax.scatter(
                    [x],
                    [yy],
                    s=s,
                    c=[nlp],
                    cmap="OrRd",
                    vmin=lo,
                    vmax=hi,
                    edgecolors="black",
                    linewidths=0.3,
                    zorder=3,
                )
            elif allow_ns:
                ax.scatter(
                    [x],
                    [yy],
                    s=8,
                    facecolors="none",
                    edgecolors="#cccccc",
                    linewidths=0.4,
                    zorder=2,
                )
    ax.set_yticks([])
    for grp, disp, pe, pi in rows:
        ax.text(
            _LABEL_X,
            ypos[disp],
            disp,
            fontsize=S.FONT_TICK - 0.5,
            va="center",
            ha="right",
        )
    for grp, ys in grp_rows.items():
        y0, y1 = min(ys) - 0.5, max(ys) + 0.5
        ax.add_patch(
            FancyBboxPatch(
                (_STRIP_X, y0),
                _STRIP_W,
                y1 - y0,
                boxstyle="round,pad=0,rounding_size=0.03",
                transform=ax.transData,
                facecolor="#F0F0F0",
                edgecolor="#cccccc",
                linewidth=0.4,
                clip_on=False,
                zorder=1,
            )
        )
        ax.text(
            _STRIP_X + _STRIP_W / 2,
            np.mean(ys),
            grp,
            fontsize=S.FONT_TICK - 1.5,
            fontweight="bold",
            color=gcol.get(grp, "#555"),
            va="center",
            ha="center",
            rotation=90,
            linespacing=0.85,
        )
    ax.set_xticks([_EXC_X, _INH_X])
    ax.set_xticklabels(["EXC", "INH"], fontsize=S.FONT_LABEL)
    for t, c in zip(
        ax.get_xticklabels(),
        [S.paper_palette_lineage["excitatory"], S.paper_palette_lineage["inhibitory"]],
    ):
        t.set_color(c)
        t.set_fontweight("bold")
    ax.set_xlim(*_XLIM)
    ax.set_ylim(-0.7, ymax - 0.3)
    ax.invert_yaxis()
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    sm = plt.cm.ScalarMappable(cmap="OrRd", norm=plt.Normalize(lo, hi))
    if short_cbar:
        cax = ax.inset_axes(
            [1.03, 0.0, 0.055, 0.42]
        )  # short bar, lower-right, outside the dotplot
        cb = fig.colorbar(sm, cax=cax)
    else:
        cb = fig.colorbar(sm, ax=ax, fraction=0.045, pad=0.02)
    cb.set_label(r"$-\log_{10}(p_{\mathrm{adj}})$", fontsize=S.FONT_TICK - 0.5)
    cb.ax.tick_params(labelsize=S.FONT_TICK - 1)
    cb.outline.set_linewidth(0.3)
    if allow_ns:
        ax.legend(
            handles=[
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    color="w",
                    markerfacecolor="none",
                    markeredgecolor="#bbb",
                    markeredgewidth=0.5,
                    markersize=4,
                    label="n.s. (padj>=0.05)",
                )
            ],
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.02),
            handletextpad=0.2,
        )
    fig.subplots_adjust(left=0.31, right=0.87, top=0.97, bottom=0.14)
    S.save_and_show(fig, save, figure=figure)


# resized treatment-convergence panel (theme groups; every term significant -> no open circles)
o = pd.read_csv(f"{KB}/enrichment_discovery_top_ora.csv")
o = o[o["kind"] == "ORA"] if "kind" in o.columns else o


def _hit(substr, lin):
    sub = o[
        (o["lineage"] == lin)
        & (o["Description"].str.contains(substr, case=False, regex=False))
    ]
    return (
        float(sub.nsmallest(1, "p.adjust").iloc[0]["p.adjust"]) if len(sub) else np.nan
    )


FIG3B = [
    ("Translation", "cytoplasmic transl.", "cytoplasmic translation"),
    ("Translation", "transl. initiation", "Cap-dependent Translation Initiation"),
    ("Translation", "transl. elongation", "Eukaryotic Translation Elongation"),
    ("Translation", "ribosome", "Ribosome"),
    ("Growth", "Myc targets V1", "MYC_TARGETS_V1"),
    ("mRNA\ndecay", "NMD", "Nonsense-Mediated Decay (NMD)"),
]
rows3b = [
    (g, d, _hit(sub, "excitatory"), _hit(sub, "inhibitory")) for g, d, sub in FIG3B
]
_convergence_dotplot(
    rows3b, {}, "fig3b_convergence_ora__half_quarter", "Fig3", allow_ns=False
)

In [ ]:
# treatment convergence / divergence (chisq ORA) - same tight layout
CD = pd.read_csv(f"{KB}/convergence_divergence_ora_discovery.csv")


def _pj(desc):
    r = CD[CD["Description"] == desc]
    return (
        (np.nan, np.nan)
        if not len(r)
        else (float(r.iloc[0]["padj_exc"]), float(r.iloc[0]["padj_inh"]))
    )


# groups follow the class of each term in convergence_divergence_ora_discovery.csv
CONVDIV = [
    ("Convergent", "transl. initiation", "Cap-dependent Translation Initiation"),
    ("Convergent", "transl. elongation", "Eukaryotic Translation Elongation"),
    ("Convergent", "cytoplasmic transl.", "cytoplasmic translation"),
    ("Convergent", "NMD / mRNA decay", "Nonsense-Mediated Decay (NMD)"),
    ("Convergent", "axon guidance", "Axon guidance"),
    ("Convergent", "RHO -> PKNs", "RHO GTPases activate PKNs"),
    (
        "Excitatory-\nspecific",
        "NMDA-R activation",
        "Activation of NMDA receptors and postsynaptic events",
    ),
    (
        "Excitatory-\nspecific",
        "Sema3A repulsion",
        "Sema3A PAK dependent Axon repulsion",
    ),
    (
        "Excitatory-\nspecific",
        "GF-R signalling",
        "Diseases of signal transduction by growth factor receptors and second messengers",
    ),
    ("Inhibitory-\nspecific", "Myc targets V1", "HALLMARK_MYC_TARGETS_V1"),
    ("Inhibitory-\nspecific", "L1CAM interactions", "L1CAM interactions"),
]
rows_cd = [(g, d, *_pj(desc)) for g, d, desc in CONVDIV]
gcol = {
    "Convergent": "#6a51a3",
    "Excitatory-\nspecific": S.paper_palette_lineage["excitatory"],
    "Inhibitory-\nspecific": S.paper_palette_lineage["inhibitory"],
}
_convergence_dotplot(
    rows_cd, gcol, "fig3b_convergence_divergence", "Fig3", allow_ns=True
)

### Variant: shortened -log10(padj) colorbar (lower-right)


In [ ]:
# --- convergence/divergence with a SHORTENED colorbar (lower-right, outside the dotplot) ---
# same panel as fig3b_convergence_divergence but the -log10(padj) colourbar is reduced in height
# and moved to the lower-right (matches the corrmat short-colorbar treatment). reuses rows_cd/gcol.
_convergence_dotplot(
    rows_cd,
    gcol,
    "fig3b_convergence_divergence__shortcbar",
    "Fig3",
    allow_ns=True,
    short_cbar=True,
)